# Programación Avanzada para el Análisis de Datos

**Proyecto Integrador:** Análisis y Modelado de Siniestros Viales en CABA

* **Alumnos:**
  * Falco Cristian Eric
  * Ortega Felix Eduardo
  * Srebernich Laura Elena
* **Profesor:** Cifuentes Duran Juan Carlos
* **Carrera:** Licenciatura en Ciencia de Datos

---

# 1. Importación de Bibliotecas y Carga de Datos

El primer paso del pipeline consiste en la ingesta de los datos crudos. Trabajaremos con dos fuentes de información proporcionadas por las autoridades de seguridad vial:
1. **Dataset de Hechos:** Contiene la información a nivel de accidente (ubicación, fecha, condiciones climáticas, etc.).
2. **Dataset de Víctimas:** Contiene el detalle granular a nivel de persona involucrada (edad, sexo, rol, gravedad de sus lesiones).

Para que el modelo pueda aprovechar el contexto completo, nuestro objetivo será unificar ambas fuentes en un único *Dataframe* maestro.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
print("--- 1. CARGA DE DATASETS ---")

# Dataset 1: Víctimas (Nivel detalle por persona)
url_victimas = 'https://raw.githubusercontent.com/Cristian-Eric-Falco/siniestros_viales_datos/refs/heads/main/siniestros_viales_victimas.csv'
df_victimas = pd.read_csv(url_victimas)
print(f"Dataset Víctimas cargado: {df_victimas.shape}")

# Dataset 2: Hechos (Nivel detalle por accidente)
url_hechos = 'https://raw.githubusercontent.com/Cristian-Eric-Falco/siniestros_viales_datos/refs/heads/main/siniestros_viales_hechos.csv'
df_hechos = pd.read_csv(url_hechos)
print(f"Dataset Hechos cargado: {df_hechos.shape}")

--- 1. CARGA DE DATASETS ---
Dataset Víctimas cargado: (62076, 9)
Dataset Hechos cargado: (54064, 21)


# 2. Preparación de Claves de Cruce (Sanidad Estructural)

Antes de realizar cualquier fusión matemática (*Merge*), es imperativo garantizar la sanidad de las columnas que actuarán como puente entre ambas tablas (claves primarias).

Si intentamos cruzar bases de datos gubernamentales sin una limpieza previa, es común perder información vital debido a espacios invisibles, diferencias de mayúsculas/minúsculas o formatos de fecha incompatibles. Para ello, definimos una función modular que estandariza estas claves.

In [ ]:
def preparar_claves_cruce(df, col_id='id_siniestro', col_fecha='fecha_siniestro'):
    """
    Limpia espacios invisibles en los IDs y convierte las fechas a formato
    datetime matemático para que el cruce sea exacto.
    """
    df_clean = df.copy()

    # 1. Limpieza de ID (Aseguramos mayúsculas y borramos espacios en blanco)
    if col_id in df_clean.columns:
        df_clean[col_id] = df_clean[col_id].astype(str).str.strip().str.upper()

    # 2. Conversión de Fechas
    if col_fecha in df_clean.columns:
        df_clean[col_fecha] = pd.to_datetime(df_clean[col_fecha], errors='coerce')

    return df_clean

print("--- 2. PREPARANDO CLAVES DE CRUCE ---")
# Aplicamos la función a ambos datasets
df_vic_prep = preparar_claves_cruce(df_victimas)
df_hechos_prep = preparar_claves_cruce(df_hechos)
print("IDs sin espacios y fechas convertidas a datetime exitosamente.")

--- 2. PREPARANDO CLAVES DE CRUCE ---
IDs sin espacios y fechas convertidas a datetime exitosamente.


# 3. Auditoría de Integridad y Validación de Negocio

El cruce de información requiere una validación lógica. En los siniestros viales, existe una aparente contradicción documentada: un accidente puede estar clasificado globalmente como `MORTAL` (porque hubo al menos un fallecido), pero individuos específicos dentro de ese mismo accidente pueden tener una clasificación `LEVE`.

El siguiente bloque audita que los IDs coincidan correctamente y demuestra la existencia de estos "sobrevivientes", validando que no se trata de un error de datos, sino de la cruda realidad de la dinámica de los accidentes de tránsito.

In [ ]:
def auditar_cruce_datasets(df_base, df_extra, id_col='id_siniestro'):
    """
    Cruza dos datasets por su ID y realiza un diagnóstico de
    contradicciones en la columna de fecha.
    """
    # Cruzamos (Inner Join) para ver qué tienen en común
    df_cruce = pd.merge(df_base, df_extra, on=id_col, how='inner', suffixes=('_VIC', '_HECHO'))

    print("=== 3. REPORTE DE INTEGRIDAD DEL CRUCE ===")
    print(f"Total registros Víctimas (Base): {len(df_base)}")
    print(f"Total registros Hechos (Extra):  {len(df_extra)}")
    print(f"IDs que hicieron 'Match':        {len(df_cruce)}\n")

    if len(df_cruce) == 0:
        print("🚨 CRÍTICO: Ningún ID coincidió.")
        return None

    # Prueba de Fuego: Contradicciones en la Fecha
    col_fecha_base = 'fecha_siniestro_VIC'
    col_fecha_extra = 'fecha_siniestro_HECHO'

    if col_fecha_base in df_cruce.columns and col_fecha_extra in df_cruce.columns:
        # Buscamos donde las fechas NO son iguales (ignora los nulos por seguridad)
        fechas_distintas = df_cruce[
            (df_cruce[col_fecha_base] != df_cruce[col_fecha_extra]) &
            (df_cruce[col_fecha_base].notna()) &
            (df_cruce[col_fecha_extra].notna())
        ]
        print(f"⚠️ Contradicciones en FECHA: {len(fechas_distintas)} registros difieren en su fecha exacta.")
        if len(fechas_distintas) > 0:
            display(fechas_distintas[[id_col, col_fecha_base, col_fecha_extra]].head())
    else:
        print("✅ Las columnas de fecha no están duplicadas o no se encontraron para comparar.")

    return df_cruce

# Ejecutamos la auditoría sobre los datos pre-lavados
df_auditoria = auditar_cruce_datasets(df_vic_prep, df_hechos_prep)

=== 3. REPORTE DE INTEGRIDAD DEL CRUCE ===
Total registros Víctimas (Base): 62076
Total registros Hechos (Extra):  54064
IDs que hicieron 'Match':        62076

⚠️ Contradicciones en FECHA: 0 registros difieren en su fecha exacta.


In [ ]:
# Comparamos la gravedad entre ambos datasets
df_cruce = pd.merge(df_vic_prep, df_hechos_prep, on='id_siniestro', how='inner', suffixes=('_VIC', '_HECHO'))

# Nota: Ajusta 'gravedad_siniestro_HECHO' si el nombre en el segundo dataset es distinto
if 'GRAVEdad_victima' in df_cruce.columns and 'gravedad_siniestro' in df_cruce.columns:
    contradicciones_grav = df_cruce[df_cruce['GRAVEdad_victima'] != df_cruce['gravedad_siniestro']]
    print(f"--- VALIDACIÓN DE GRAVEDAD ---")
    print(f"Registros con gravedad distinta: {len(contradicciones_grav)}")

    if len(contradicciones_grav) > 0:
        display(contradicciones_grav[['id_siniestro', 'GRAVEdad_victima', 'gravedad_siniestro']].head())
    else:
        print("✅ ¡Perfecto! La gravedad coincide en todos los registros cruzados.")

--- VALIDACIÓN DE GRAVEDAD ---
Registros con gravedad distinta: 470


,id_siniestro,GRAVEdad_victima,gravedad_siniestro
1490,HC-2019-0108835,LEVE,MORTAL
1511,HC-2019-0108835,LEVE,MORTAL
1893,HC-2019-0133232,LEVE,MORTAL
1896,HC-2019-0133232,LEVE,MORTAL
1897,HC-2019-0133232,LEVE,MORTAL


A simple vista parece que los datos están corruptos, pero te aseguro al 100% que no es un error, es la cruda realidad de cómo funcionan los accidentes de tránsito.

Veamos un ejemplo. Fíjate muy bien en los IDs:
El siniestro HC-2019-0108835 aparece en la fila 1490 y se repite en la 1511. El siniestro HC-2019-0133232 aparece tres veces seguidas (1893, 1896, 1897).

¿Qué está pasando aquí? La diferencia entre "El Hecho" y "La Víctima"
gravedad_siniestro (La tabla de Hechos): Califica el accidente como un todo. Por regla policial, un accidente adopta la etiqueta de la peor consecuencia. Si en un choque múltiple hay 5 personas ilesas y 1 fallecido, el accidente entero se clasifica como MORTAL.

GRAVEdad_victima (La tabla de Víctimas): Califica la salud individual de cada persona.

Tus 470 "contradicciones" son en realidad las historias de los sobrevivientes. Esos registros nos dicen que en esos accidentes catalogados como mortales, hubo 470 personas (pasajeros, el otro conductor, peatones cercanos) que lograron salir con heridas LEVES.

# 4. Fusión Definitiva y Feature Engineering

Aprobada la auditoría, procedemos a realizar un `Left Join`. Tomamos la tabla de Víctimas como nuestra base (ya que buscamos predecir el destino individual de cada persona) y le acoplamos el contexto del entorno (Tabla de Hechos).

Además, aplicamos ingeniería de características (*Feature Engineering*). Transformar la hora exacta del accidente en "Franjas Horarias" permite al modelo detectar agrupaciones de riesgo (como el peligro inherente a la madrugada) evitando el ruido estadístico de los minutos exactos.

In [ ]:
print("--- 4. FUSIÓN FINAL Y FEATURE ENGINEERING ---")

# 1. Unimos los datasets (Mantenemos todas las víctimas y traemos info del hecho)
# Eliminamos columnas repetidas y la gravedad del HECHO para evitar confusiones y fugas de datos
columnas_a_excluir = ['id_siniestro', 'gravedad_siniestro']
cols_extra = [col for col in df_hechos_prep.columns if col not in df_vic_prep.columns or col in columnas_a_excluir]

# Usamos 'left' join para mantener todas las víctimas
df_final = pd.merge(df_vic_prep, df_hechos_prep[cols_extra], on='id_siniestro', how='left')

# 2. Aplicamos la Franja Horaria (Nuestro predictor estrella)
def categorizar_franja_horaria(hora):
    if pd.isna(hora): return 'Desconocido'
    hora = int(hora)
    if 6 <= hora < 12: return '1. Mañana (06h-12h)'
    elif 12 <= hora < 18: return '2. Tarde (12h-18h)'
    elif 18 <= hora < 24: return '3. Noche (18h-24h)'
    else: return '4. Madrugada (00h-06h)'

# Creamos la franja horaria
if 'rango_horario' in df_final.columns:
    df_final['franja_horaria'] = df_final['rango_horario'].apply(categorizar_franja_horaria)

# 3. Limpiamos la columna objetivo
# Aprovechamos para estandarizar el nombre a minúsculas si estaba rara
df_final.rename(columns={'GRAVEdad_victima': 'gravedad_victima'}, inplace=True)

print(f"✅ Dataset Maestro creado.")
print(f"   Filas iniciales de víctimas: {len(df_vic_prep)}")
print(f"   Filas en dataset final:      {len(df_final)} (Deben ser iguales)")

display(df_final.head())

--- 4. FUSIÓN FINAL Y FEATURE ENGINEERING ---
✅ Dataset Maestro creado.
   Filas iniciales de víctimas: 62076
   Filas en dataset final:      62076 (Deben ser iguales)


,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,gravedad_victima,rol_victima,fecha_fallecimiento_victima,numero_total_de_victimas,...,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,contraparte_siniestro,gravedad_siniestro,franja_horaria
0,LC-2019-0000647,2019-01-01,2019,MOTO,M,54,GRAVE,SD,NaN,1,...,DE LOS CONSTITUYENTES AV. y HABANA,Comuna 12,AVENIDA,POINT (97510.27137648081406951 105087.83369881...,-58.490436,-34.583403,MOTO-AUTO,AUTO,GRAVE,3. Noche (18h-24h)
1,LC-2019-0000600,2019-01-01,2019,SD,F,1,LEVE,SD,NaN,1,...,NaN,Comuna 1,NaN,POINT (108178.21037640585564077 104229.8809107...,-58.374154,-34.591108,PEATON-TRANSPORTE PUBLICO,TRANSPORTE PUBLICO,LEVE,3. Noche (18h-24h)
2,LC-2019-0000136,2019-01-01,2019,SD,F,21,LEVE,SD,NaN,2,...,NaN,Comuna 15,NaN,POINT (102193.35632690557395108 104565.9966998...,-58.439392,-34.588108,MOTO-AUTO,AUTO,LEVE,1. Mañana (06h-12h)
3,LC-2019-0000082,2019-01-01,2019,SD,F,32,LEVE,SD,NaN,4,...,NaN,Comuna 3,NaN,POINT (105968.98286849579017144 102737.1734686...,-58.398225,-34.604579,AUTO-TRANSPORTE PUBLICO,TRANSPORTE PUBLICO,LEVE,4. Madrugada (00h-06h)
4,LC-2019-0000194,2019-01-01,2019,SD,F,33,LEVE,SD,NaN,1,...,NaN,Comuna 9,NaN,POINT (94030.76669932194636203 97681.071761248...,-58.528413,-34.650156,AUTO-CAMION,CAMION,LEVE,1. Mañana (06h-12h)


Tratamiento valores nulos

# 5. Estandarización de Nulos y Prevención de Data Leakage

En los registros policiales, es habitual encontrar campos marcados como "SD" (Sin Dato). Si no normalizamos estas cadenas de texto a un formato nulo real matemático (`np.nan`), los algoritmos de imputación no podrán procesarlos posteriormente.

Finalmente, eliminamos columnas que representan un peligro de **Data Leakage (Fuga de Datos)**. Variables como `fecha_fallecimiento_victima` o los contadores de heridos (`numero_victimas_mortal...`) le darían al modelo la respuesta por adelantado de manera tramposa. Un modelo predictivo en el mundo real no tiene acceso al certificado de defunción al momento del choque, por lo que estas columnas deben ser erradicadas antes del entrenamiento.

In [ ]:
import numpy as np

print("--- 5. LIMPIEZA FINAL: ESTANDARIZACIÓN DE VALORES 'SD' ---")

# Creamos nuestro dataset definitivo para el modelo
df_modelo = df_final.copy()

# 1. Definimos todas las formas en que puede estar escrito "Sin Dato"
variantes_sd = ['SD', 'sd', 'Sd', 'S.D.', 's.d.', 'ND', 'nd', 'Nd', 'N.D.']

# 2. Identificamos automáticamente cuáles son las columnas de texto
columnas_texto = df_modelo.select_dtypes(include=['object', 'category']).columns

# 3. Barremos columna por columna aplicando el reemplazo
for col in columnas_texto:
    # Usamos replace para cambiar esas palabras exactas por un nulo real de Numpy
    df_modelo[col] = df_modelo[col].replace(variantes_sd, np.nan)

# 4. Auditoría post-limpieza
print("✅ Limpieza ejecutada con éxito.")
print("\nTop 10 columnas con más valores nulos reales (NaN) listos para tratar:")

# Contamos los nulos, los ordenamos de mayor a menor y mostramos los 10 peores
nulos_totales = df_modelo.isna().sum().sort_values(ascending=False)
display(nulos_totales[nulos_totales > 0].head(10))

--- 5. LIMPIEZA FINAL: ESTANDARIZACIÓN DE VALORES 'SD' ---
✅ Limpieza ejecutada con éxito.

Top 10 columnas con más valores nulos reales (NaN) listos para tratar:


fecha_fallecimiento_victima        61466
rol_victima                        50203
modo_desplazamiento_victima        21874
edad_victima                       16756
direccion_normalizada_siniestro    15022
contraparte_siniestro              14387
tipo_de_via_siniestro              14209
sexo_victima                       11054
comuna_siniestro                    3387
latitud_siniestro                   2728
dtype: int64

Se eliminan columnas sin poder predictivo y columnas que pueden producir data lakeage en los modelos

In [ ]:
# Eliminar columnas específicas
columnas_a_eliminar = [
    'id_siniestro',
    'fecha_siniestro',
    'fecha_fallecimiento_victima',
    'direccion_normalizada_siniestro',
    'latitud_siniestro',
    'longitud_siniestro',
    'geocodificacion_plana',
    'gravedad_siniestro',
    'numero_victimas_leve_siniestro',
    'numero_victimas_grave_siniestro',
    'numero_victimas_mortal_siniestro'
]

df_modelo = df_modelo.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"Columnas eliminadas. Shape actual: {df_modelo.shape}")

Columnas eliminadas. Shape actual: (62076, 16)


In [ ]:
display(df_modelo.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62076 entries, 0 to 62075
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   anio_siniestro               62076 non-null  int64  
 1   modo_desplazamiento_victima  40202 non-null  object 
 2   sexo_victima                 51022 non-null  object 
 3   edad_victima                 45320 non-null  object 
 4   gravedad_victima             62076 non-null  object 
 5   rol_victima                  11873 non-null  object 
 6   numero_total_de_victimas     62076 non-null  int64  
 7   mes_siniestro                62076 non-null  int64  
 8   dia_siniestro                62076 non-null  int64  
 9   hora_siniestro               61984 non-null  object 
 10  rango_horario                61984 non-null  float64
 11  comuna_siniestro             58689 non-null  object 
 12  tipo_de_via_siniestro        47867 non-null  object 
 13  participantes_si

None

# 5. Modelado Predictivo y Evaluación

Tras la fase de limpieza e ingeniería de características, procedemos a la construcción del modelo predictivo. Dada la naturaleza de los datos (variables categóricas, no linealidades) y la presencia de algunos valores nulos inherentes a la recolección policial, se ha decidido trabajar exclusivamente con **modelos basados en árboles de decisión (Random Forest y XGBoost)**. Estos algoritmos son altamente eficientes, manejando la estructura tabular de manera óptima.

## 5.1. Metodología de Evaluación (Stratified K-Fold Cross-Validation)
El principal desafío de este proyecto es el **extremo desbalanceo de clases**: la clase `MORTAL` representa apenas el ~1% de los registros.
Una simple división de entrenamiento y prueba (Train/Test Split) podría generar resultados engañosos por mero azar estadístico. Para garantizar la robustez académica y operativa de los resultados, implementaremos una **Validación Cruzada Estratificada de 5 particiones (5-Fold Stratified CV)**.

Además, para evitar la fuga de datos (*Data Leakage*), cualquier transformación (como la imputación de nulos) se realizará de manera dinámica **dentro** de cada ciclo de evaluación, utilizando únicamente la información del set de entrenamiento en curso.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("--- CONFIGURACIÓN DEL ENTORNO DE MODELADO Y EVALUACIÓN ---")

# 1. Separación de X e y
columnas_a_ignorar = ['id_siniestro', 'fecha_siniestro', 'gravedad_victima']
X = df_modelo.drop(columns=[col for col in columnas_a_ignorar if col in df_modelo.columns])
y = df_modelo['gravedad_victima']

# 2. Codificación del Objetivo (Label Encoding)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
diccionario_clases = dict(zip(le.transform(le.classes_), le.classes_))
nombres_clases = [diccionario_clases[i] for i in range(len(diccionario_clases))]
print(f"Clases a predecir: {diccionario_clases}")

# 3. Codificación de Variables Categóricas
columnas_texto = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in columnas_texto:
    le_col = LabelEncoder()
    X[col] = X[col].fillna('MISSING')
    X[col] = le_col.fit_transform(X[col])

print(f"Variables predictoras listas: {X.shape[1]} columnas.\n")

# ==========================================
# 4. MOTOR MAESTRO DE EVALUACIÓN (CON MÉTRICAS COMPLETAS)
# ==========================================
def evaluar_modelo_cv(X_data, y_data, modelo, nombre_modelo, umbral_mortal=None, umbral_grave=None, usar_pesos_v2=False):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    resultados = defaultdict(list)

    print(f"🔄 Evaluando: {nombre_modelo}")

    for fold, (train_index, test_index) in enumerate(skf.split(X_data, y_data), 1):
        X_tr, X_te = X_data.iloc[train_index].copy(), X_data.iloc[test_index].copy()
        y_tr, y_te = y_data[train_index].copy(), y_data[test_index].copy()

        imputer = SimpleImputer(strategy='median')
        X_tr = pd.DataFrame(imputer.fit_transform(X_tr), columns=X_tr.columns)
        X_te = pd.DataFrame(imputer.transform(X_te), columns=X_te.columns)

        pesos_tr = None
        if usar_pesos_v2:
            idx_grave = list(le.classes_).index('GRAVE')
            idx_mortal = list(le.classes_).index('MORTAL')
            pesos_tr = np.ones(len(y_tr))
            pesos_tr[y_tr == idx_grave] = 3.0
            pesos_tr[y_tr == idx_mortal] = 5.0
            modelo.fit(X_tr, y_tr, sample_weight=pesos_tr)
        else:
            modelo.fit(X_tr, y_tr)

        if umbral_mortal is not None and umbral_grave is not None:
            probabilidades = modelo.predict_proba(X_te)
            idx_grave = list(le.classes_).index('GRAVE')
            idx_mortal = list(le.classes_).index('MORTAL')

            y_pred = modelo.predict(X_te).copy()
            y_pred[(probabilidades[:, idx_grave] >= umbral_grave) & (probabilidades[:, idx_mortal] < umbral_mortal)] = idx_grave
            y_pred[probabilidades[:, idx_mortal] >= umbral_mortal] = idx_mortal
        else:
            y_pred = modelo.predict(X_te)

        # Extraemos Reporte Completo
        reporte = classification_report(y_te, y_pred, target_names=nombres_clases, output_dict=True, zero_division=0)

        # Guardamos Recall y Precision
        resultados['recall_grave'].append(reporte['GRAVE']['recall'])
        resultados['precision_grave'].append(reporte['GRAVE']['precision'])

        resultados['recall_mortal'].append(reporte['MORTAL']['recall'])
        resultados['precision_mortal'].append(reporte['MORTAL']['precision'])

        resultados['f1_weighted'].append(reporte['weighted avg']['f1-score'])

    # Imprimir Resumen Promediado
    print(f"   [Promedios 5-Folds]")
    print(f"   MORTAL -> Recall: {np.mean(resultados['recall_mortal']):.3f} (±{np.std(resultados['recall_mortal']):.3f}) | Precision: {np.mean(resultados['precision_mortal']):.3f}")
    print(f"   GRAVE  -> Recall: {np.mean(resultados['recall_grave']):.3f} (±{np.std(resultados['recall_grave']):.3f}) | Precision: {np.mean(resultados['precision_grave']):.3f}")
    print(f"   F1-Weighted Global: {np.mean(resultados['f1_weighted']):.3f}\n")

    return modelo

print("✅ Motor de evaluación actualizado con métricas completas.")

--- CONFIGURACIÓN DEL ENTORNO DE MODELADO Y EVALUACIÓN ---
Clases a predecir: {np.int64(0): 'GRAVE', np.int64(1): 'LEVE', np.int64(2): 'MORTAL'}
Variables predictoras listas: 15 columnas.

✅ Motor de evaluación actualizado con métricas completas.


## 5.2. Línea Base 1: Random Forest Classifier

Para establecer una primera línea base de rendimiento (*baseline*), utilizaremos un modelo de **Random Forest**. Este algoritmo de ensamble (basado en *bagging*) es robusto frente a valores atípicos y no requiere escalado de variables, lo que lo convierte en un excelente punto de partida para datos tabulares.

Al ejecutarlo en su configuración estándar (sin ajuste de pesos ni umbrales), observaremos cómo un modelo tradicional se enfrenta al desbalanceo extremo de nuestra clase objetivo (`MORTAL`).

In [ ]:
print("--- 5.2 LÍNEA BASE 1: RANDOM FOREST ESTÁNDAR ---")

# Instanciamos el modelo base
# n_jobs=-1 permite usar todos los núcleos del procesador para mayor velocidad
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Evaluamos con nuestro motor de validación cruzada
# No pasamos pesos ni umbrales, es el comportamiento por defecto
modelo_rf = evaluar_modelo_cv(X, pd.Series(y_encoded),
                              modelo=rf_baseline,
                              nombre_modelo="Random Forest (Estándar)")

--- 5.2 LÍNEA BASE 1: RANDOM FOREST ESTÁNDAR ---
🔄 Evaluando: Random Forest (Estándar)
   [Promedios 5-Folds]
   MORTAL -> Recall: 0.557 (±0.026) | Precision: 0.971
   GRAVE  -> Recall: 0.263 (±0.028) | Precision: 0.793
   F1-Weighted Global: 0.955



## 5.3. Línea Base 2: XGBoost Estándar

Como segunda línea base, implementaremos **XGBoost** (Extreme Gradient Boosting). A diferencia del Random Forest, XGBoost construye árboles de forma secuencial, donde cada nuevo árbol intenta corregir los errores de los anteriores (*boosting*).

Evaluaremos este algoritmo en su configuración predeterminada. Es esperable que alcance una precisión global (*Accuracy*) engañosamente alta, pero que fracase al intentar capturar la minoría de casos fatales (*Recall Mortal*), demostrando la necesidad de aplicar técnicas avanzadas de sensibilidad al costo.

In [ ]:
print("--- 5.3 LÍNEA BASE 2: XGBOOST ESTÁNDAR ---")

# Instanciamos el modelo XGBoost base
# tree_method='hist' es altamente eficiente para datasets grandes y maneja nulos nativamente
xgb_baseline = xgb.XGBClassifier(
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    eval_metric='mlogloss',
    max_depth=6 # Profundidad estándar
)

# Evaluamos con nuestro motor de validación cruzada
modelo_xgb_base = evaluar_modelo_cv(X, pd.Series(y_encoded),
                                    modelo=xgb_baseline,
                                    nombre_modelo="XGBoost (Estándar)")

--- 5.3 LÍNEA BASE 2: XGBOOST ESTÁNDAR ---
🔄 Evaluando: XGBoost (Estándar)
   [Promedios 5-Folds]
   MORTAL -> Recall: 0.595 (±0.015) | Precision: 0.920
   GRAVE  -> Recall: 0.277 (±0.022) | Precision: 0.780
   F1-Weighted Global: 0.956



## 5.3. Decisión Estratégica: El Valor de la Prevención

Los resultados de las líneas base confirman un problema operativo crítico: los modelos estándar son altamente conservadores. Maximizan la precisión general a costa de ignorar las fatalidades, alcanzando tasas de detección (Recall) inaceptables para un entorno de emergencia médica.

Dada la naturaleza de la **Seguridad Vial y la Gestión de Emergencias**, se determina que **el costo de un falso negativo (no predecir una muerte o herida grave) es infinitamente superior al de un falso positivo (generar una alerta preventiva)**.

**Acciones a tomar en la V2:**
1.  **Aprendizaje Sensible al Costo:** Se asignarán pesos asimétricos durante el entrenamiento para penalizar severamente los errores en las clases `MORTAL` (5x) y `GRAVE` (3x).
2.  **Ajuste de Umbral (Threshold Moving):** Se reducirán los umbrales de decisión, exigiendo menor "seguridad" matemática al modelo para disparar una alerta crítica.

**Objetivo esperado:**
*   Incrementar el **Recall de la clase MORTAL por encima del 80%**.
*   Rescatar a la clase `GRAVE` (la zona gris) mejorando su detección simultáneamente.
*   Aceptar una reducción controlada en la Precisión (tolerando falsas alarmas, es decir, despachar ambulancias de alta complejidad de forma preventiva).

In [ ]:
print("--- 5.4 DIAGNÓSTICO: El problema de la clase GRAVE ---")

# Hacemos un split rápido (solo para este análisis diagnóstico)
from sklearn.model_selection import train_test_split
X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(X, pd.Series(y_encoded), test_size=0.2, random_state=42, stratify=y_encoded)

# Imputamos rápidamente
imputer_diag = SimpleImputer(strategy='median')
X_train_diag = pd.DataFrame(imputer_diag.fit_transform(X_train_diag), columns=X.columns)
X_test_diag = pd.DataFrame(imputer_diag.transform(X_test_diag), columns=X.columns)

# Entrenamos el modelo base para inspeccionarlo
xgb_baseline.fit(X_train_diag, y_train_diag)
probabilidades = xgb_baseline.predict_proba(X_test_diag)

idx_leve = list(le.classes_).index('LEVE')
idx_grave = list(le.classes_).index('GRAVE')
idx_mortal = list(le.classes_).index('MORTAL')

print("\n📊 DISTRIBUCIÓN DE CONFIANZAS PROMEDIO POR CLASE REAL:")
print(f"   LEVE real   -> Confianza del modelo: {probabilidades[y_test_diag==idx_leve, idx_leve].mean():.3f}")
print(f"   MORTAL real -> Confianza del modelo: {probabilidades[y_test_diag==idx_mortal, idx_mortal].mean():.3f}")
print(f"   GRAVE real  -> Confianza del modelo: {probabilidades[y_test_diag==idx_grave, idx_grave].mean():.3f} ⚠️ (Muy baja)")

# Matriz de confusión para ver a dónde van los GRAVES
y_pred_diag = xgb_baseline.predict(X_test_diag)
cm_diag = confusion_matrix(y_test_diag, y_pred_diag, labels=[idx_leve, idx_grave, idx_mortal])
grave_reales = cm_diag[1, :]
total_grave = grave_reales.sum()

print(f"\n🎯 ¿A dónde predice la clase GRAVE el modelo estándar?")
print(f"De {total_grave} accidentes GRAVE reales:")
print(f"   → {grave_reales[0]} ({grave_reales[0]/total_grave*100:.1f}%) se confunden y predicen como LEVE")
print(f"   → {grave_reales[1]} ({grave_reales[1]/total_grave*100:.1f}%) se predicen correctamente como GRAVE")
print(f"   → {grave_reales[2]} ({grave_reales[2]/total_grave*100:.1f}%) se predicen como MORTAL")

print("\n💡 CONCLUSIÓN: La clase GRAVE carece de fronteras claras en los datos viales y es absorbida constantemente por la clase mayoritaria (LEVE).")

--- 5.4 DIAGNÓSTICO: El problema de la clase GRAVE ---

📊 DISTRIBUCIÓN DE CONFIANZAS PROMEDIO POR CLASE REAL:
   LEVE real   -> Confianza del modelo: 0.974
   MORTAL real -> Confianza del modelo: 0.589
   GRAVE real  -> Confianza del modelo: 0.292 ⚠️ (Muy baja)

🎯 ¿A dónde predice la clase GRAVE el modelo estándar?
De 494 accidentes GRAVE reales:
   → 371 (75.1%) se confunden y predicen como LEVE
   → 119 (24.1%) se predicen correctamente como GRAVE
   → 4 (0.8%) se predicen como MORTAL

💡 CONCLUSIÓN: La clase GRAVE carece de fronteras claras en los datos viales y es absorbida constantemente por la clase mayoritaria (LEVE).


## 5.5. Optimización: Búsqueda del Umbral Dual (Dual Threshold)

Habiendo justificado la necesidad de maximizar el Recall, procedemos a optimizar nuestro modelo XGBoost.

Primero, implementaremos **Pesos Diferenciales** (`sample_weight`) durante el entrenamiento para indicarle al modelo la jerarquía de importancia (MORTAL: 5.0x, GRAVE: 3.0x, LEVE: 1.0x).

Luego, en lugar de aceptar la decisión estándar del modelo (donde gana la clase con mayor probabilidad absoluta), implementaremos una regla de **Umbral Dual**. Evaluaremos múltiples combinaciones de umbrales para las probabilidades de las clases `GRAVE` y `MORTAL` con el objetivo de encontrar el punto de equilibrio operativo perfecto.

In [ ]:
print("--- 5.5 BÚSQUEDA DE UMBRALES ÓPTIMOS (DUAL THRESHOLD) ---")

# Entrenamos un modelo V2 temporal solo para la búsqueda de umbrales
xgb_v2_opt = xgb.XGBClassifier(
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    eval_metric='mlogloss',
    max_depth=7,
    learning_rate=0.05,
    n_estimators=200,
    reg_alpha=0.5,
    reg_lambda=1.0
)

# Imputación rápida para la búsqueda
imputer_opt = SimpleImputer(strategy='median')
X_train_opt = pd.DataFrame(imputer_opt.fit_transform(X_train_diag), columns=X.columns)
X_test_opt = pd.DataFrame(imputer_opt.transform(X_test_diag), columns=X.columns)

# Aplicación de Pesos (Cost-Sensitive Learning)
pesos_opt = np.ones(len(y_train_diag))
pesos_opt[y_train_diag == idx_grave] = 3.0
pesos_opt[y_train_diag == idx_mortal] = 5.0

xgb_v2_opt.fit(X_train_opt, y_train_diag, sample_weight=pesos_opt, verbose=False)
probabilidades_opt = xgb_v2_opt.predict_proba(X_test_opt)

# Definimos pares de umbrales a evaluar (Umbral GRAVE, Umbral MORTAL)
umbrales_combinations = [
    (0.30, 0.10),
    (0.25, 0.10),
    (0.20, 0.10),
    (0.15, 0.10),
    (0.20, 0.15),
    (0.20, 0.20),
]

resultados_opt = []

print("🔍 Evaluando pares de umbrales en set de prueba (Formato: G / M):")

for umbral_g, umbral_m in umbrales_combinations:
    y_pred_opt = xgb_v2_opt.predict(X_test_opt).copy()

    # Lógica de Umbral Dual
    y_pred_opt[(probabilidades_opt[:, idx_grave] >= umbral_g) & (probabilidades_opt[:, idx_mortal] < umbral_m)] = idx_grave
    y_pred_opt[probabilidades_opt[:, idx_mortal] >= umbral_m] = idx_mortal

    report_opt = classification_report(y_test_diag, y_pred_opt, target_names=nombres_clases, output_dict=True)

    resultados_opt.append({
        'u_grave': umbral_g,
        'u_mortal': umbral_m,
        'rec_grave': report_opt['GRAVE']['recall'],
        'rec_mortal': report_opt['MORTAL']['recall']
    })

    print(f"   ({umbral_g:.2f}, {umbral_m:.2f}) → Recall GRAVE: {report_opt['GRAVE']['recall']:.2f} | Recall MORTAL: {report_opt['MORTAL']['recall']:.2f}")

# Selección automática del mejor balance
mejor_combo = max(resultados_opt, key=lambda x: x['rec_grave'] + x['rec_mortal'])
UMBRAL_GRAVE_FINAL = mejor_combo['u_grave']
UMBRAL_MORTAL_FINAL = mejor_combo['u_mortal']

print(f"\n✅ Umbrales Óptimos Seleccionados para Producción:")
print(f"   Umbral GRAVE: {UMBRAL_GRAVE_FINAL:.2f}")
print(f"   Umbral MORTAL: {UMBRAL_MORTAL_FINAL:.2f}")

--- 5.5 BÚSQUEDA DE UMBRALES ÓPTIMOS (DUAL THRESHOLD) ---
🔍 Evaluando pares de umbrales en set de prueba (Formato: G / M):
   (0.30, 0.10) → Recall GRAVE: 0.37 | Recall MORTAL: 0.86
   (0.25, 0.10) → Recall GRAVE: 0.42 | Recall MORTAL: 0.86
   (0.20, 0.10) → Recall GRAVE: 0.51 | Recall MORTAL: 0.86
   (0.15, 0.10) → Recall GRAVE: 0.62 | Recall MORTAL: 0.86
   (0.20, 0.15) → Recall GRAVE: 0.56 | Recall MORTAL: 0.82
   (0.20, 0.20) → Recall GRAVE: 0.59 | Recall MORTAL: 0.76

✅ Umbrales Óptimos Seleccionados para Producción:
   Umbral GRAVE: 0.15
   Umbral MORTAL: 0.10


## 5.6. Evaluación del Modelo Final: XGBoost V2 (Pesos + Dual Threshold)

Habiendo identificado los umbrales operativos ideales (GRAVE: 0.15, MORTAL: 0.10) y habiendo integrado el aprendizaje sensible al costo, procedemos a evaluar nuestro **Modelo V2** definitivo utilizando la **Validación Cruzada Estratificada de 5 particiones**.

Este es el test final que determinará la estabilidad y eficacia del algoritmo bajo condiciones rigurosas, garantizando que su rendimiento no es producto del azar en la división de los datos.

In [ ]:
print("--- 5.6 EVALUACIÓN FINAL: XGBOOST V2 (PESOS + DUAL THRESHOLD) ---")

# Instanciamos el modelo V2 (parámetros ajustados)
xgb_v2_final = xgb.XGBClassifier(
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    eval_metric='mlogloss',
    max_depth=7,
    learning_rate=0.05,
    n_estimators=200,
    reg_alpha=0.5,
    reg_lambda=1.0
)

# Evaluamos utilizando el Motor Maestro
# Activamos la bandera 'usar_pesos_v2' y pasamos los umbrales óptimos
modelo_final = evaluar_modelo_cv(
    X, pd.Series(y_encoded),
    modelo=xgb_v2_final,
    nombre_modelo="XGBoost V2 (Dual Threshold)",
    umbral_mortal=UMBRAL_MORTAL_FINAL,
    umbral_grave=UMBRAL_GRAVE_FINAL,
    usar_pesos_v2=True
)

print("\n🏆 CONCLUSIÓN DE MODELADO:")
print("El Modelo V2 logra el equilibrio operativo perfecto, detectando consistentemente")
print("al >80% de los casos Mortales y al >65% de los casos Graves en validación cruzada,")
print("sin necesidad de fabricar datos sintéticos, manteniendo la integridad del dataset policial.")

--- 5.6 EVALUACIÓN FINAL: XGBOOST V2 (PESOS + DUAL THRESHOLD) ---
🔄 Evaluando: XGBoost V2 (Dual Threshold)
   [Promedios 5-Folds]
   MORTAL -> Recall: 0.849 (±0.030) | Precision: 0.228
   GRAVE  -> Recall: 0.659 (±0.013) | Precision: 0.171
   F1-Weighted Global: 0.878


🏆 CONCLUSIÓN DE MODELADO:
El Modelo V2 logra el equilibrio operativo perfecto, detectando consistentemente
al >80% de los casos Mortales y al >65% de los casos Graves en validación cruzada,
sin necesidad de fabricar datos sintéticos, manteniendo la integridad del dataset policial.
